# TBP — Primera prueba con series reales de Argentina

Este notebook calcula el TBP por cohorte (1950–2035) con las series de `datos/procesados/` (ver `datos/README.md`). Es una **prueba exploratoria**: sirve para ver si los datos se comportan, no para concluir.

**Qué es dato y qué es supuesto** (todo queda a la vista):

| Variable | Valor | Origen | Tipo |
|---|---|---|---|
| $N_t$ | nacimientos | DEIS (≤2024); ONU proyección (2025+) | observado / proyectado |
| $1-\mu$ | supervivencia de la **cohorte** hasta 65 | ONU WPP 2024, tabla de mortalidad (producto de $p_x$ por edad y año calendario) | estimado |
| $J_{t+65}$ | población 65+ en $t+65$ | ONU WPP 2024 | proyectado |
| $E_r$ | esperanza de vida a los 65 en $t+65$ | ONU WPP 2024 (de período) | proyectado |
| $\tau$ | 11 % personal + 10,77 % patronal = **21,77 %** | Anuario ANSES 2008–2023, p. 39 | normativa (sin aportes a PAMI/obra social) |
| $\rho$ | **40,3 %** (haber medio SIPA / salario imponible medio, dic-2023) | Anuario ANSES, p. 91 | observado, **constante** |
| $\bar A$ | **14,2** (obs.) o **30** (legal) años | ANSES p. 46 (varones 64 años: 15,8; mujeres 59 años: 12,6; promedio simple) / requisito legal | **supuesto — escenario** |

> ⚠️ $\bar A$ observado está **sesgado a la baja**: ANSES solo cuenta aportes desde jul-1994 y a personas con ≥12 meses de aporte. El 30 es el requisito legal (techo). La verdad está entre ambos. **Por eso el nivel del TBP no es interpretable todavía; la forma de la curva y su tendencia sí.**

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
try: display
except NameError: display = print

# --- Carga de datos -------------------------------------------------------
# Orden de busqueda: 1) carpeta del repo (../datos/procesados), 2) Drive (opcional), 3) descarga desde GitHub.
# En Colab con Drive: descomentar las dos lineas siguientes para montarlo.
# from google.colab import drive; drive.mount('/content/drive')
ARCHIVOS = ["series_tbp_extraidas.csv", "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"]
# >>> REEMPLAZAR USUARIO/REPO por el repositorio real al conocerlo <<<
RAW_BASE = "https://raw.githubusercontent.com/USUARIO/REPO/main/datos/procesados"
CANDIDATOS = ["../datos/procesados", "datos/procesados",
              "/content/drive/MyDrive/JUANDE/CURSAR/06. Taller de indicadores/datos_tbp",
              "/content/drive/MyDrive/datos_tbp", "/content/datos_tbp", "."]
DATA = next((p for p in CANDIDATOS if os.path.exists(os.path.join(p, "series_tbp_extraidas.csv"))), None)
if DATA is None:   # no hay copia local: descargar desde GitHub (raw)
    DATA = "datos_tbp_descargados"; os.makedirs(DATA, exist_ok=True)
    for f in ARCHIVOS:
        urllib.request.urlretrieve(f"{RAW_BASE}/{f}", os.path.join(DATA, f))
print("Datos en:", os.path.abspath(DATA))

# Las salidas (CSV y PNG) se escriben en ./salidas_tbp para no pisar los datos versionados.
OUT = "salidas_tbp"; os.makedirs(OUT, exist_ok=True)

s = pd.read_csv(f"{DATA}/series_tbp_extraidas.csv").set_index("year")
mort = pd.read_csv(f"{DATA}/un_wpp2024_argentina_tabla_mortalidad_abreviada.csv", low_memory=False)
mort = mort[mort.Sex == "Total"]
display(s.loc[[1950, 1990, 2010, 2024, 2040, 2091], ["deis_nacimientos", "un_births_miles", "un_e65", "un_pob_65mas_miles"]])

## 1. Supervivencia de la cohorte (1−μ)

Para la cohorte nacida en $t$ se multiplica la probabilidad de sobrevivir cada grupo de edad ($0, 1\text{–}4, 5\text{–}9,\dots,60\text{–}64$) usando la tabla de mortalidad del **año calendario en que ese grupo vive**. Es más correcto que usar la tabla del año de nacimiento ("de período"), que ignora que la mortalidad va bajando.

In [ ]:
px = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
grp = mort[mort.Time == 2000].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]]
grp = grp[grp.AgeGrpStart < 65]

def surv65_cohorte(t):
    S = 1.0
    for a, sp in grp.itertuples(index=False):
        y = int(np.floor(t + a + sp/2))
        if y > px.index.max() or y < px.index.min():
            return np.nan
        S *= px.loc[y, a]
    return S

print("Ejemplo cohorte 2000: 1-mu =", round(surv65_cohorte(2000), 4), "| tabla de período del año 2000 (ignora mejoras futuras):", round(s.un_S65_periodo[2000], 4))

## 2. Cálculo por cohorte

$\tau$ y $\rho$ son constantes (los datos de ANSES no dan serie anual utilizable, ver `anses_valores_extraidos.csv`). **Todo el movimiento de la curva viene de la demografía** ($N_t$, $1-\mu$, $J$, $E_r$).

In [ ]:
TAU = 0.11 + 0.1077     # Anuario ANSES p.39 (aporte personal 11% + contribución patronal SIPA 10,77% actividades inc. a)
RHO = 0.403             # Anuario ANSES p.91: haber medio SIPA / salario imponible medio, dic-2023
A_OBS, A_LEGAL = 14.2, 30.0

rows = []
for t in range(1950, 2036):
    y = t + 65
    N_deis = s.deis_nacimientos.get(t, np.nan)
    N_un = s.un_births_miles[t] * 1000
    rows.append(dict(cohorte=t, anio_65=y,
        N_t=N_deis if not np.isnan(N_deis) else N_un,
        fuente_N="DEIS" if not np.isnan(N_deis) else "ONU proyectado",
        N_un=N_un, S65=surv65_cohorte(t),
        J=s.un_pob_65mas_miles[y]*1000, E65=s.un_e65[y], pob_edad65=s.un_pob_edad65_miles[y]*1000))
d = pd.DataFrame(rows)

for nom, A in [("obs", A_OBS), ("legal", A_LEGAL)]:
    d[f"TBP_A_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*d.E65*RHO)     # fórmula original del grupo
    d[f"TBP_C_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*RHO)           # en años (sin E_r)
    d[f"TBP_B_{nom}"] = A*TAU/(d.E65*RHO)                     # cohorte pura, sin N
d["D"] = d.N_t*d.S65/d.J                                       # factor demográfico

d.to_csv(f"{OUT}/tbp_argentina_cohortes_1950_2035.csv", index=False)
display(d[d.cohorte.isin([1950,1975,1990,2000,2010,2015,2020,2024,2030,2035])][
    ["cohorte","N_t","fuente_N","S65","J","E65","D","TBP_A_obs","TBP_A_legal","TBP_C_legal","TBP_B_legal"]])

## 3. Grupo de control con datos reales

Tres chequeos que **no dependen** de los supuestos de $\bar A$, $\tau$ ni $\rho$:

1. **Identidades algebraicas:** $TBP_A = D\cdot B$ y $TBP_A = TBP_C/E_r$.
2. **Consistencia demográfica:** la población de exactamente 65 años en $t+65$ (ONU) debería ser ≈ $N_t \cdot (1-\mu)$ (nacidos que sobreviven; la diferencia es migración y error de las fuentes).
3. **Consistencia entre fuentes:** nacimientos DEIS vs ONU.

In [ ]:
# 1) identidades
B = d.TBP_B_legal
assert np.allclose(d.TBP_A_legal, d.D * B)
assert np.allclose(d.TBP_A_legal, d.TBP_C_legal / d.E65)
print("✅ Identidades TBP_A = D×B y TBP_A = TBP_C/E_r: OK en las 86 cohortes")

# 2) cohorte reconstruida vs población observada/proyectada a los 65
d["ratio_pob65_vs_NS"] = d.pob_edad65 / (d.N_t * d.S65)
r = d.ratio_pob65_vs_NS
print(f"\n2) población(65 años en t+65) / (N_t × S65): media {r.mean():.3f} | min {r.min():.3f} | max {r.max():.3f}")
fuera = d[(r < 0.85) | (r > 1.20)][["cohorte", "fuente_N", "ratio_pob65_vs_NS"]]
print("   Cohortes fuera de [0.85, 1.20]:", "ninguna" if fuera.empty else "")
if not fuera.empty: display(fuera)

# 3) DEIS vs ONU
c = d[d.fuente_N == "DEIS"].copy()
c["ONU/DEIS"] = c.N_un / c.N_t
print(f"\n3) Nacimientos ONU/DEIS ({c.cohorte.min()}–{c.cohorte.max()}): media {c['ONU/DEIS'].mean():.3f} | min {c['ONU/DEIS'].min():.3f} | max {c['ONU/DEIS'].max():.3f}")
display(c.loc[c.cohorte.isin([1950,1980,2000,2015,2020,2022,2023,2024]), ["cohorte", "N_t", "N_un", "ONU/DEIS"]])

## 4. Gráficos

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(d.cohorte, d.N_t/1e3, label="Usado (DEIS ≤2024, ONU después)")
ax[0].plot(d.cohorte, d.N_un/1e3, "--", label="ONU WPP 2024")
ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Nacimientos (miles)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(d.cohorte, d.D, color="tab:green")
ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("D = N·(1−μ) / J₆₅₊ (factor demográfico)"); ax[1].grid(alpha=.3)

ax[2].plot(d.cohorte, d.TBP_A_legal, label="TBP_A  (Ā = 30, techo)")
ax[2].plot(d.cohorte, d.TBP_A_obs, label="TBP_A  (Ā = 14,2, piso sesgado)")
ax[2].plot(d.cohorte, d.TBP_B_legal, "--", label="TBP_B (cohorte pura, Ā = 30)")
ax[2].axhline(1, color="gray", ls=":"); ax[2].axvline(2024.5, color="gray", ls=":")
ax[2].set_title("TBP por cohorte"); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel("año de nacimiento de la cohorte")
plt.tight_layout(); plt.savefig(f"{OUT}/tbp_argentina_graficos.png", dpi=130); plt.show()

## 5. Qué muestran los datos (lectura preliminar)

* **Caída de la natalidad real:** de ~756 mil (2010) a 413 mil (2024). La cohorte 2024 tiene el factor demográfico $D$ más bajo de toda la serie.
* **Rebote artificial después de 2024:** la ONU proyecta ~507 mil nacimientos en 2024 y ~520 mil hacia 2030. Eso **no refleja** la caída registrada por DEIS. El repunte de las cohortes 2025–2035 es un efecto del supuesto ONU, **no un dato**: hay que reemplazarlo con un escenario propio de natalidad.
* **Longevidad:** $E_{65}$ pasa de 17,9 (cohorte 1950) a ~24,6 (cohorte 2035). Eso empuja el TBP hacia abajo por dos vías (en $J$ y en $E_r$), el problema de doble conteo detectado en la auditoría.
* **La curva de $TBP_B$** (sin natalidad) baja suavemente por longevidad; la de $TBP_A$ baja mucho más y tiene la caída de natalidad encima. La diferencia entre ambas es el aporte de la natalidad.
* **Todavía no se puede afirmar:** niveles absolutos ($\bar A$ incierto), umbral "TBP=1", ni proyecciones más allá de 2024 basadas en natalidad ONU.